# Mutlimodal Training goes brr

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import pickle
import re
import string

/home/gagan/Desktop/side-projects/amazon-mlchallenge/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class Config:
    # Paths
    TRAIN_EMBEDDINGS_PATH = "train_embeddings.npy"
    TRAIN_IDS_PATH = "train_image_ids.pkl"
    TEST_EMBEDDINGS_PATH = "test_embeddings.npy"
    TEST_IDS_PATH = "test_image_ids.pkl"
    
    TRAIN_CSV = "../../train_processed.csv"
    TEST_CSV = "../../test_processed.csv"
    
    # Model settings0
    
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    BATCH_SIZE = 128
    LEARNING_RATE = 1e-3
    EPOCHS = 100
    PATIENCE = 10
    
    # Architecture
    TEXT_DIM = 768  # BERT embedding dimension
    IMAGE_DIM = 2048  # ResNet50 embedding dimension (or 512 for CLIP ViT-B/32)
    HIDDEN_DIM = 512
    DROPOUT = 0.3


In [10]:

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================
def load_embeddings(embeddings_path, ids_path):
    """Load saved embeddings and IDs"""
    embeddings = np.load(embeddings_path)
    with open(ids_path, 'rb') as f:
        ids = pickle.load(f)
    return embeddings, ids


def clean_text(text):
    """Clean text data"""
    if pd.isnull(text):
        return ""
    text = re.sub(r'[\U00010000-\U0010ffff]', '', text)
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def get_bert_embeddings(texts, tokenizer, model, batch_size=32, max_length=128, device='cuda'):
    """Extract BERT embeddings from text"""
    embeddings = []
    model.eval()
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting text embeddings"):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)
        
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)
    
    return np.vstack(embeddings)


def create_image_embedding_map(image_embeddings, image_ids):
    """Create a mapping from image filename to embedding"""
    embedding_map = {}
    for i, img_id in enumerate(image_ids):
        # Handle both with and without .jpg extension
        embedding_map[img_id] = image_embeddings[i]
        if not img_id.endswith('.jpg'):
            embedding_map[img_id + '.jpg'] = image_embeddings[i]
    return embedding_map


def align_image_embeddings(df, embedding_map, default_dim):
    """Align image embeddings with dataframe rows"""
    aligned_embeddings = []
    missing_count = 0
    
    for idx, row in df.iterrows():
        img_name = row['image_name']
        
        # Try exact match
        if img_name in embedding_map:
            aligned_embeddings.append(embedding_map[img_name])
        # Try without extension
        elif img_name.replace('.jpg', '') in embedding_map:
            aligned_embeddings.append(embedding_map[img_name.replace('.jpg', '')])
        else:
            # Use zero vector for missing images
            aligned_embeddings.append(np.zeros(default_dim))
            missing_count += 1
    
    print(f"Missing image embeddings: {missing_count}/{len(df)}")
    return np.array(aligned_embeddings)


In [11]:

# ============================================================================
# ADVANCED MULTIMODAL MODEL
# ============================================================================
class MultimodalFusionModel(nn.Module):
    """Advanced multimodal model with attention-based fusion"""
    
    def __init__(self, text_dim, image_dim, value_dim=1, hidden_dim=512, dropout=0.3):
        super().__init__()
        
        # Text pathway
        self.text_encoder = nn.Sequential(
            nn.Linear(text_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Image pathway
        self.image_encoder = nn.Sequential(
            nn.Linear(image_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Value pathway
        self.value_encoder = nn.Sequential(
            nn.Linear(value_dim, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Cross-modal attention
        self.text_to_image_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim // 2,
            num_heads=8,
            dropout=dropout,
            batch_first=True
        )
        
        self.image_to_text_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim // 2,
            num_heads=8,
            dropout=dropout,
            batch_first=True
        )
        
        # Fusion layer
        fusion_input_dim = (hidden_dim // 2) * 2 + 64  # text + image + value
        self.fusion = nn.Sequential(
            nn.Linear(fusion_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(128, 1)
        )
    
    def forward(self, text_features, image_features, value_features):
        # Encode each modality
        text_encoded = self.text_encoder(text_features)
        image_encoded = self.image_encoder(image_features)
        value_encoded = self.value_encoder(value_features)
        
        # Apply cross-modal attention
        text_encoded_unsq = text_encoded.unsqueeze(1)
        image_encoded_unsq = image_encoded.unsqueeze(1)
        
        # Text attending to image
        text_attended, _ = self.text_to_image_attn(
            text_encoded_unsq, image_encoded_unsq, image_encoded_unsq
        )
        text_attended = text_attended.squeeze(1)
        
        # Image attending to text
        image_attended, _ = self.image_to_text_attn(
            image_encoded_unsq, text_encoded_unsq, text_encoded_unsq
        )
        image_attended = image_attended.squeeze(1)
        
        # Combine with residual connections
        text_final = text_encoded + text_attended
        image_final = image_encoded + image_attended
        
        # Concatenate all features
        combined = torch.cat([text_final, image_final, value_encoded], dim=1)
        
        # Final prediction
        output = self.fusion(combined)
        return output.squeeze(-1)


class SimpleMultimodalModel(nn.Module):
    """Simpler multimodal model without attention (faster, still effective)"""
    
    def __init__(self, text_dim, image_dim, value_dim=1, hidden_dim=512, dropout=0.3):
        super().__init__()
        
        combined_dim = text_dim + image_dim + value_dim
        
        self.network = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(hidden_dim // 2, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            
            nn.Linear(128, 1)
        )
    
    def forward(self, text_features, image_features, value_features):
        combined = torch.cat([text_features, image_features, value_features], dim=1)
        return self.network(combined).squeeze(-1)



In [12]:

# ============================================================================
# DATASET CLASS
# ============================================================================
class MultimodalDataset(Dataset):
    def __init__(self, text_emb, image_emb, value, targets=None):
        self.text_emb = torch.tensor(text_emb, dtype=torch.float32)
        self.image_emb = torch.tensor(image_emb, dtype=torch.float32)
        self.value = torch.tensor(value, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32) if targets is not None else None
    
    def __len__(self):
        return len(self.text_emb)
    
    def __getitem__(self, idx):
        if self.targets is not None:
            return self.text_emb[idx], self.image_emb[idx], self.value[idx], self.targets[idx]
        return self.text_emb[idx], self.image_emb[idx], self.value[idx]



In [13]:

# ============================================================================
# TRAINING FUNCTIONS
# ============================================================================
def train_epoch(model, dataloader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0
    
    for text, image, value, targets in dataloader:
        text = text.to(device)
        image = image.to(device)
        value = value.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()
        predictions = model(text, image, value)
        loss = loss_fn(predictions, targets)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)


def validate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for text, image, value, targets in dataloader:
            text = text.to(device)
            image = image.to(device)
            value = value.to(device)
            targets = targets.to(device)
            
            predictions = model(text, image, value)
            loss = loss_fn(predictions, targets)
            
            total_loss += loss.item()
            all_preds.append(predictions.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
    
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    
    return total_loss / len(dataloader), all_preds, all_targets


def smape(y_true, y_pred):
    """Calculate SMAPE metric"""
    y_true = np.expm1(y_true)
    y_pred = np.expm1(y_pred)
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))


In [2]:

# ============================================================================
# MAIN EXECUTION
# ============================================================================
def main():
    config = Config()
    device = torch.device(config.DEVICE)
    print(f"Using device: {device}")
    
    # ========================================================================
    # 1. LOAD DATA
    # ========================================================================
    print("\n" + "="*80)
    print("LOADING DATA")
    print("="*80)
    
    train_df = pd.read_csv(config.TRAIN_CSV)
    test_df = pd.read_csv(config.TEST_CSV)
    
    print(f"Train samples: {len(train_df)}")
    print(f"Test samples: {len(test_df)}")
    
    # ========================================================================
    # 2. LOAD IMAGE EMBEDDINGS
    # ========================================================================
    print("\n" + "="*80)
    print("LOADING IMAGE EMBEDDINGS")
    print("="*80)
    
    train_img_embeddings, train_img_ids = load_embeddings(
        config.TRAIN_EMBEDDINGS_PATH,
        config.TRAIN_IDS_PATH
    )
    test_img_embeddings, test_img_ids = load_embeddings(
        config.TEST_EMBEDDINGS_PATH,
        config.TEST_IDS_PATH
    )
    
    print(f"Train image embeddings shape: {train_img_embeddings.shape}")
    print(f"Test image embeddings shape: {test_img_embeddings.shape}")
    
    # Update IMAGE_DIM based on actual embeddings
    config.IMAGE_DIM = train_img_embeddings.shape[1]
    print(f"Image embedding dimension: {config.IMAGE_DIM}")
    
    # Create embedding maps
    train_img_map = create_image_embedding_map(train_img_embeddings, train_img_ids)
    test_img_map = create_image_embedding_map(test_img_embeddings, test_img_ids)
    
    # ========================================================================
    # 3. PREPARE TEXT DATA
    # ========================================================================
    print("\n" + "="*80)
    print("PREPARING TEXT DATA")
    print("="*80)
    
    # Clean text
    train_df['model_input'] = train_df['model_input'].apply(clean_text)
    test_df['model_input'] = test_df['model_input'].fillna('').apply(clean_text)
    
    # Split train into train/val
    X = train_df[['sample_id', 'value', 'model_input', 'image_name']]
    y = train_df['log_price']
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    print(f"Train split: {len(X_train)}")
    print(f"Val split: {len(X_val)}")
    
    # ========================================================================
    # 4. EXTRACT TEXT EMBEDDINGS
    # ========================================================================
    print("\n" + "="*80)
    print("EXTRACTING TEXT EMBEDDINGS")
    print("="*80)
    
    tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased')
    bert_model = AutoModel.from_pretrained('google-bert/bert-base-uncased')
    bert_model.to(device)
    bert_model.eval()
    
    X_train_text_emb = get_bert_embeddings(
        X_train['model_input'].tolist(), tokenizer, bert_model, device=device
    )
    X_val_text_emb = get_bert_embeddings(
        X_val['model_input'].tolist(), tokenizer, bert_model, device=device
    )
    X_test_text_emb = get_bert_embeddings(
        test_df['model_input'].tolist(), tokenizer, bert_model, device=device
    )
    
    print(f"Train text embeddings: {X_train_text_emb.shape}")
    print(f"Val text embeddings: {X_val_text_emb.shape}")
    print(f"Test text embeddings: {X_test_text_emb.shape}")
    
    # ========================================================================
    # 5. ALIGN IMAGE EMBEDDINGS
    # ========================================================================
    print("\n" + "="*80)
    print("ALIGNING IMAGE EMBEDDINGS")
    print("="*80)
    
    X_train_img_emb = align_image_embeddings(X_train, train_img_map, config.IMAGE_DIM)
    X_val_img_emb = align_image_embeddings(X_val, train_img_map, config.IMAGE_DIM)
    X_test_img_emb = align_image_embeddings(test_df, test_img_map, config.IMAGE_DIM)
    
    # ========================================================================
    # 6. PREPARE VALUE FEATURES
    # ========================================================================
    print("\n" + "="*80)
    print("SCALING VALUE FEATURES")
    print("="*80)
    
    scaler = StandardScaler()
    X_train_value = scaler.fit_transform(X_train[['value']].values)
    X_val_value = scaler.transform(X_val[['value']].values)
    X_test_value = scaler.transform(test_df[['value']].values)
    
    # ========================================================================
    # 7. CREATE DATASETS AND DATALOADERS
    # ========================================================================
    print("\n" + "="*80)
    print("CREATING DATASETS")
    print("="*80)
    
    train_dataset = MultimodalDataset(
        X_train_text_emb, X_train_img_emb, X_train_value, y_train.values
    )
    val_dataset = MultimodalDataset(
        X_val_text_emb, X_val_img_emb, X_val_value, y_val.values
    )
    
    train_loader = DataLoader(
        train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=2
    )
    val_loader = DataLoader(
        val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=2
    )
    
    # ========================================================================
    # 8. INITIALIZE MODEL
    # ========================================================================
    print("\n" + "="*80)
    print("INITIALIZING MODEL")
    print("="*80)
    
    # Choose model architecture
    # Option 1: Advanced fusion model with attention
    model = MultimodalFusionModel(
        text_dim=config.TEXT_DIM,
        image_dim=config.IMAGE_DIM,
        value_dim=1,
        hidden_dim=config.HIDDEN_DIM,
        dropout=config.DROPOUT
    ).to(device)
    
    # Option 2: Simple concatenation model (uncomment to use)
    # model = SimpleMultimodalModel(
    #     text_dim=config.TEXT_DIM,
    #     image_dim=config.IMAGE_DIM,
    #     value_dim=1,
    #     hidden_dim=config.HIDDEN_DIM,
    #     dropout=config.DROPOUT
    # ).to(device)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )
    loss_fn = nn.MSELoss()
    
    # ========================================================================
    # 9. TRAINING LOOP
    # ========================================================================
    print("\n" + "="*80)
    print("TRAINING MODEL")
    print("="*80)
    
    best_val_loss = float('inf')
    best_smape = float('inf')
    patience_counter = 0
    
    for epoch in range(config.EPOCHS):
        train_loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
        val_loss, val_preds, val_targets = validate(model, val_loader, loss_fn, device)
        val_smape = smape(val_targets, val_preds)
        
        scheduler.step(val_loss)
        
        print(f"Epoch {epoch+1}/{config.EPOCHS}")
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss: {val_loss:.4f}")
        print(f"  Val SMAPE: {val_smape:.2f}%")
        
        # Save best model
        if val_smape < best_smape:
            best_smape = val_smape
            best_val_loss = val_loss
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'val_smape': val_smape,
            }, 'best_multimodal_model.pt')
            print(f"  ✓ New best model saved! (SMAPE: {val_smape:.2f}%)")
        else:
            patience_counter += 1
            if patience_counter >= config.PATIENCE:
                print(f"\nEarly stopping triggered after {epoch+1} epochs")
                break
    
    # ========================================================================
    # 10. LOAD BEST MODEL AND MAKE PREDICTIONS
    # ========================================================================
    print("\n" + "="*80)
    print("GENERATING PREDICTIONS")
    print("="*80)
    
    checkpoint = torch.load('best_multimodal_model.pt')
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded best model from epoch {checkpoint['epoch']+1}")
    print(f"Best validation SMAPE: {checkpoint['val_smape']:.2f}%")
    
    # Create test dataset
    test_dataset = MultimodalDataset(
        X_test_text_emb, X_test_img_emb, X_test_value
    )
    test_loader = DataLoader(
        test_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=2
    )
    
    # Make predictions
    model.eval()
    test_preds = []
    with torch.no_grad():
        for text, image, value in test_loader:
            text = text.to(device)
            image = image.to(device)
            value = value.to(device)
            preds = model(text, image, value)
            test_preds.append(preds.cpu().numpy())
    
    test_preds = np.concatenate(test_preds)
    
    # Convert from log scale to original scale
    test_prices = np.expm1(test_preds).clip(0)
    
    # ========================================================================
    # 11. CREATE SUBMISSION
    # ========================================================================
    submission = pd.DataFrame({
        'sample_id': test_df['sample_id'],
        'price': test_prices
    })
    
    submission.to_csv('submission_multimodal.csv', index=False)
    print("\n" + "="*80)
    print("SUBMISSION SAVED")
    print("="*80)
    print(f"File: submission_multimodal.csv")
    print(f"Samples: {len(submission)}")
    print(f"Price range: [{submission['price'].min():.2f}, {submission['price'].max():.2f}]")
    print(f"Mean price: {submission['price'].mean():.2f}")

In [17]:
main()

Using device: cuda

LOADING DATA
Train samples: 75000
Test samples: 75000

LOADING IMAGE EMBEDDINGS


: 